In [ ]:
# 1. Install a verified published V2 commit; local checkout and ZIP are optional.
import io, importlib, stat, subprocess, sys, tempfile, zipfile
from pathlib import Path
REPOSITORY = "https://github.com/Cheva1234/TamaBench.git"
REF = "baf81456053e736031a0c46da9ae1e782d020500"  # Published V2; pin an exact reviewed commit for reproducibility.
LOCAL_CHECKOUT = ""  # Optional local source. Takes priority over REF; clear both for ZIP upload.
if LOCAL_CHECKOUT:
    source = LOCAL_CHECKOUT
elif REF:
    source = f"git+{REPOSITORY}@{REF}"
else:
    from google.colab import files
    uploaded = files.upload()  # Select the reviewed TamaBench V2 source ZIP.
    if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith(".zip"):
        raise ValueError("Upload exactly one TamaBench source ZIP")
    target = Path(tempfile.mkdtemp(prefix="tamabench-source-"))
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        entries = archive.infolist()
        if len(entries) > 2000 or sum(item.file_size for item in entries) > 64 * 1024 * 1024:
            raise ValueError("Source ZIP exceeds the safety limit")
        for item in entries:
            name = item.filename.replace("\\", "/")
            destination = (target / name).resolve()
            if (name.startswith("/") or ".." in Path(name).parts
                    or not destination.is_relative_to(target.resolve())
                    or stat.S_ISLNK(item.external_attr >> 16)):
                raise ValueError("Unsafe path or symlink in source ZIP")
        archive.extractall(target)
    projects = list(target.rglob("pyproject.toml"))
    if len(projects) != 1:
        raise ValueError("Source ZIP must contain exactly one TamaBench checkout")
    source = str(projects[0].parent)
if (LOCAL_CHECKOUT or not REF) and not (Path(source) / "pyproject.toml").is_file():
    raise ValueError("Local source must contain pyproject.toml")
subprocess.check_call([sys.executable, "-m", "pip", "install", source])
importlib.invalidate_caches()
from tamabench.config import RunConfig
assert RunConfig().benchmark_version == "2.0.0", "Selected source does not contain TamaBench V2"


In [ ]:
#@title 2. Choose CPU or API provider (no GPU required for hosted APIs)
import os
from tamabench.config import RunConfig
from tamabench.experiment import doctor

PROVIDER = "cpu" #@param ["cpu", "openai", "openrouter", "groq", "ollama", "custom"]
MODEL = "" #@param {type:"string"}
# Exact model ID from your provider; no cloud model or spending is chosen for you.
API_BASE = "" #@param {type:"string"}
# Set only for custom or an alternative Ollama server. Use the base, not /chat/completions.
API_KEY_ENV = "" #@param {type:"string"}
# Optional name override, NEVER the key itself. Presets choose the normal secret name.
CUSTOM_AUTH_REQUIRED = True #@param {type:"boolean"}
USE_DRIVE = False #@param {type:"boolean"}
EPISODES = 1 #@param {type:"integer"}
HORIZON_MINUTES = 240 #@param {type:"integer"}
MAX_API_CALLS = 30 #@param {type:"integer"}
MAX_OUTPUT_TOKENS = 512 #@param {type:"integer"}
MAX_TOTAL_TOKENS = 30000 #@param {type:"integer"}
MAX_WALL_SECONDS = 300 #@param {type:"integer"}
OUTPUT_DIR = "/content/tamabench-output"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/tamabench-output"
options = {}
if PROVIDER != "cpu":
    if not MODEL.strip():
        raise ValueError("Set MODEL to your provider's exact model ID before running.")
    options.update(provider=PROVIDER, model=MODEL)
    if API_BASE:
        options["api_base"] = API_BASE
    if API_KEY_ENV:
        options["api_key_env"] = API_KEY_ENV
    if PROVIDER == "custom":
        options["require_api_key"] = CUSTOM_AUTH_REQUIRED
config = RunConfig(
    **options, episodes=EPISODES, seed_start=42,
    max_simulated_minutes=HORIZON_MINUTES,
    max_api_calls=MAX_API_CALLS, max_output_tokens=MAX_OUTPUT_TOKENS,
    max_total_tokens=MAX_TOTAL_TOKENS, max_wall_seconds=MAX_WALL_SECONDS,
    max_retries=0, timeout=60.0,
    output_dir=OUTPUT_DIR, display="compact", resume=True,
)
if config.require_api_key and not os.environ.get(config.api_key_env, "").strip():
    try:
        from google.colab import userdata
        secret = userdata.get(config.api_key_env)
        if not isinstance(secret, str) or not secret.strip():
            raise ValueError("Empty secret")
        os.environ[config.api_key_env] = secret
        del secret
    except Exception:
        raise RuntimeError(
            f"Add {config.api_key_env} in Colab's Secrets panel and enable notebook access. "
            "Never paste your API key into a code cell."
        ) from None
check = doctor(config)  # Local validation only: sends zero requests.
print(check)
if not check["ready"]:
    raise RuntimeError("; ".join(check["errors"]))
# 240 minutes / one seed is a setup smoke test, not a full benchmark ranking.
# For comparable results use 7 * 1440, multiple seeds, and matching explicit budgets.
# Model-specific sampling/reasoning/JSON mode options can be added to RunConfig.


In [ ]:
# 3. Execute the shared experiment API; no notebook benchmark loop.
from tamabench.experiment import doctor, run_experiment
result = run_experiment(config)
print({"experiment_id": result.experiment_id, "run_ids": result.run_ids,
       "skipped_run_ids": result.skipped_run_ids, "output_dir": result.output_dir})


In [ ]:
# 4. Download the complete output bundle; Drive users also retain the original.
import shutil
from google.colab import files
archive = shutil.make_archive("/content/tamabench-results", "zip", root_dir=result.output_dir)
files.download(archive)
